# Illustration of the parallel BME algorithm

## Preparation

The script was tested and working with scikit-bio 0.7.3. Because it uses multiple private functions, their presence and behavior may not stay the same in future versions of scikit-bio. Therefore, it will be a safe measure to pin the version to 0.7.3.

In [1]:
import skbio
skbio.__version__

'0.7.3'

In [2]:
import numpy as np

In [3]:
from skbio import TreeNode

In [4]:
from skbio.tree._me import (
    _insert_taxon, _bal_insert_plan, _bal_avgdist_chunk, _bal_update_spine
)
from skbio.tree._c_me import (
    _preorder, _calc_sizes, _calc_deeps, _calc_pairs
)
from skbio.tree.tests.test_me import _check_tree

## Test data

This is a dummy dataset of 11 taxa. 10 taxa are already in the current tree ($T_{k-1}$), with the first taxon (`a`) placed at the root and the remaining 9 taxa (`b` to `j`) at leaves. Therefore, the current tree has 17 nodes in total.

The task is to insert the next taxon `k` into the tree to make it $T_k$. This will create a new leaf and a new internal node serving as the connector. The resulting tree will contain 10 leaves and 19 nodes in total.

In [5]:
taxa = list('abcdefghijk')

In [6]:
nwk = '(((1,4)3,(((2,10)9,(14,16)15)13,(8,12)11)7)5,6)0;'

In [7]:
obj = TreeNode.read([nwk])

In [8]:
print(obj.ascii_art())

                              /-1
                    /3-------|
                   |          \-4
                   |
                   |                              /-2
          /5-------|                    /9-------|
         |         |                   |          \-10
         |         |          /13------|
         |         |         |         |          /-14
         |         |         |          \15------|
-0-------|          \7-------|                    \-16
         |                   |
         |                   |          /-8
         |                    \11------|
         |                              \-12
         |
          \-6


In [9]:
n = obj.count()
n

17

In [10]:
m = obj.count(tips=True)
m

9

In [11]:
taxmap = dict(zip(range(3), range(3))) | dict(zip(range(4, 2 * m, 2), range(3, m + 1)))
taxmap

{0: 0, 1: 1, 2: 2, 4: 3, 6: 4, 8: 5, 10: 6, 12: 7, 14: 8, 16: 9}

Tree topology

In [12]:
tree = np.full((n + 2, 4), -1, dtype=int)

In [13]:
tree = np.array([
    [ 5,  6,  0,  0],
    [ 0,  1,  3,  4],
    [ 0,  2,  9, 10],
    [ 1,  4,  5,  7],
    [ 0,  3,  3,  1],
    [ 3,  7,  0,  6],
    [ 0,  4,  0,  5],
    [13, 11,  5,  3],
    [ 0,  5, 11, 12],
    [ 2, 10, 13, 15],
    [ 0,  6,  9,  2],
    [ 8, 12,  7, 13],
    [ 0,  7, 11,  8],
    [ 9, 15,  7, 11],
    [ 0,  8, 15, 16],
    [14, 16, 13,  9],
    [ 0, 10, 15, 14],
    [-1, -1, -1, -1],
    [-1, -1, -1, -1],
])

In [14]:
_check_tree(tree, n)

Preorder

In [15]:
order = np.full(n + 2, -1, dtype=int)
_preorder(order, tree, np.empty(n + 2, dtype=int))
order

array([ 0,  5,  3,  1,  4,  7, 13,  9,  2, 10, 15, 14, 16, 11,  8, 12,  6,
       -1, -1])

Clade sizes

- number of nodes (tips and internal, incl. root) within each clade (i.e., size)

In [16]:
sizes = np.full(n + 2, -1, dtype=int)
_calc_sizes(n, tree, order, sizes)
sizes

array([17,  1,  1,  3,  1, 15,  1, 11,  1,  3,  1,  3,  1,  7,  1,  3,  1,
       -1, -1])

Node depths

- number of branches connecting each node to root

In [17]:
deeps = np.full(n + 2, -1, dtype=int)
_calc_deeps(n, tree, order, deeps)
deeps

array([ 0,  3,  5,  2,  3,  1,  1,  2,  4,  4,  5,  3,  4,  3,  5,  4,  5,
       -1, -1])

Number of pairs

- Number of ancestor-descendant pairs (i.e., sum of depths) within each clade

In [18]:
pairs = np.full(n + 2, -1, dtype=int)
_calc_pairs(n, tree, order, sizes, pairs)
pairs

array([54,  0,  0,  2,  0, 38,  0, 22,  0,  2,  0,  2,  0, 10,  0,  2,  0,
       -1, -1])

Take a snapshot of the current status

In [19]:
tree0 = tree.copy()
order0 = order.copy()
sizes0 = sizes.copy()
deeps0 = deeps.copy()
pairs0 = pairs.copy()

Balanced average distances ($\delta$) between all subtrees. Each cell $(i,j)$ represents the distance between two disjoint subtrees rooted at nodes $i$ and $j$.

- This is the dominant factor of the entire BME algorithm.

In [20]:
adm = np.zeros((n + 2, n + 2), dtype=float)

Balanced average distances from an added taxon ($k$) to each subtree.

- The first row represents lower subtrees (descending from each node).
- The second row represents upper subtrees (ascending from the parent of each node)

In [21]:
adk = np.zeros((2, n + 2), dtype=float)

An array to store intermediates, primarily used to store the difference between the distance between the new taxon and each subtree $\delta_{k,x}$ and the distance between the subtree and the insertion point $\delta_{x,y_0}$.

In [22]:
diffs = np.empty(n + 2, dtype=float)

Ancestors of target (nodes from its parent to root in ascending order)

In [23]:
ancs = np.empty(n + 2, dtype=int)

Pointers to rows in `adm` representing individual ancestors

In [24]:
ancx = np.empty(n + 2, dtype=int)

Segment bounds (nodes within each segment has the same level)

In [25]:
segs = np.empty(n + 3, dtype=int)

Level (number of branches ascending from target's parent) of each segment

In [26]:
lvls = np.empty(n + 2, dtype=int)

Workload (ops) of clade under each segment start

In [27]:
oops = np.empty(n + 2, dtype=int)

## Insert new taxon

New taxon

In [28]:
k = m + 1
k

10

In [29]:
taxa[k]

'k'

New taxon's node index

In [30]:
n + 1

18

Connector's node index

In [31]:
n

17

Preorder index of the target node. We will insert the new taxon $k$ into the branch above node 13 (6 in preorder).

In [32]:
itag = 6

In [33]:
tag = order[itag]
int(tag)

13

In [34]:
size = sizes[tag]
int(size)

7

In [35]:
deep = deeps[tag]
int(deep)

3

Update balanced average distance matrix after taxon insertion

In [36]:
_bal_insert_plan(
    n, itag, adm, adk, tree, order, sizes, deeps, pairs, diffs, ancs, ancx, segs, lvls, oops, True
)

Ancestry of the target node in ascending order

In [37]:
ancs[:deep]

array([7, 5, 0])

In [38]:
ancx[:deep]

array([133,  95,   0])

Number of segments

In [39]:
n_segs = (segs == n).argmax()
n_segs.item()

6

In [40]:
segs[:n_segs + 1]

array([ 0,  1,  5,  6, 13, 16, 17])

In [41]:
lvls[:n_segs]

array([2, 1, 0, 0, 0, 2])

In [42]:
oops[:n_segs]

array([16, 12,  6,  4,  2,  2])

A helper function to calculate the number of operations per node and per clade

In [43]:
def _calc_ops(n, order, sizes, pairs, segs, lvls, oops):
    iseg = 1
    seg = segs[1]
    lvl = lvls[0]
    node_ops, tree_ops = [lvl], [oops[0]]
    for i in range(1, n):
        node = order[i]
        size = sizes[node]
        if i == seg:
            lvl = lvls[iseg]
            if i <= itag:
                node_ops.append(lvl)
                tree_ops.append(oops[iseg])
            else:
                node_ops.append(size + lvl - 1)
                tree_ops.append(pairs[node] + size * lvl)
            iseg += 1
            seg = segs[iseg]
        else:
            node_ops.append(size + lvl - 1)
            tree_ops.append(pairs[node] + size * lvl)
    return np.array(node_ops), np.array(tree_ops)

In [44]:
node_ops, tree_ops = _calc_ops(n, order, sizes, pairs, segs, lvls, oops)

In [45]:
node_ops

array([2, 1, 3, 1, 1, 0, 0, 2, 0, 0, 2, 0, 0, 2, 0, 0, 2])

In [46]:
tree_ops

array([16, 12,  5,  1,  1,  6,  4,  2,  0,  0,  2,  0,  0,  2,  0,  0,  2])

In [47]:
assert node_ops.sum() == tree_ops[0]

In [48]:
for i in range(n):
    assert tree_ops[i] == node_ops[i : i + sizes[order[i]]].sum()

## Chunk nodes

Chunk bounds in preorder

- The maximum number of chunks is n (i.e., one node per chunk), therefore n + 1 boundaries are needed. The first bound must be 0.

In [49]:
chunks = np.zeros(n + 1, dtype=int)
chunks[0] = 0

Segment index at the beginning of each chunk

In [50]:
chusegs = np.zeros(n, dtype=int)
chusegs[0] = 1

Expected number of chunks. Workload will be roughly evenly divided into these chunks. An empirical setting of this parameter is 10 $\times$ number of threads assigned to the algorithm. The actual number of chunks usually exceeds this number.

In [51]:
enc = 3

Expected chunk capacity (total workload of the tree / this number)

In [52]:
int(oops[0] + enc - 1) // enc

6

Perform chunking and return the actual number of chunks

In [53]:
nc = _bal_avgdist_chunk(
    n, itag, order, sizes, pairs, segs, lvls, oops, enc, chunks, chusegs
)
nc

3

In [54]:
chunks[:nc + 1]

array([ 0,  3, 13, 17])

In [55]:
chusegs[:nc]

array([1, 2, 4])

Workload per chunk

In [56]:
for i in range(nc):
    print(node_ops[chunks[i] : chunks[i + 1]].sum())

6
6
4


In [57]:
# expected chunking result (with enc = 3):
# order: 0,  5,  3,  1,  4,  7, 13,  9,  2, 10, 15, 14, 16, 11,  8, 12,  6

# segs: 0, 1, 5, 6, 13, 16, 17
# lvls: 2, 1, 0, -1, 0, 2
# oops: 16, 12, 6, 4, 2, 2

# chunks: 0, 3, 13, 17

# chunk 0 (ops = 6): nodes 0, 5, 3
# chunk 1 (ops = 6): nodes 1, 4, 7, 13, 9, 2, 10, 15, 14, 16
# chunk 2 (ops = 4): nodes 11, 8, 12, 6

In [58]:
#                               /-1
#                     /3-------|
#                    |          \-4
#                    |
#                    |                              /-2
#           /5-------|                    /9-------|
#          |         |                   |          \-10
#          |         |          /13------|
#          |         |         |         |          /-14
#          |         |         |          \15------|
# -0-------|          \7-------|                    \-16
#          |                   |
#          |                   |          /-8
#          |                    \11------|
#          |                              \-12
#          |
#           \-6

## Update tree structure

In [59]:
_bal_update_spine(tag, deep, sizes, pairs, ancs)

In [60]:
_insert_taxon(m + 1, itag, size, tree, order)

Updated tree topology

- Only relationships around the insertion point are updated.

In [61]:
tree

array([[ 5,  6,  0,  0],
       [ 0,  1,  3,  4],
       [ 0,  2,  9, 10],
       [ 1,  4,  5,  7],
       [ 0,  3,  3,  1],
       [ 3,  7,  0,  6],
       [ 0,  4,  0,  5],
       [17, 11,  5,  3],
       [ 0,  5, 11, 12],
       [ 2, 10, 13, 15],
       [ 0,  6,  9,  2],
       [ 8, 12,  7, 17],
       [ 0,  7, 11,  8],
       [ 9, 15, 17, 18],
       [ 0,  8, 15, 16],
       [14, 16, 13,  9],
       [ 0, 10, 15, 14],
       [13, 18,  7, 11],
       [ 0, 10, 17, 13]])

Mark changed cells in the tree topology

In [62]:
(tree0[:-2] != tree[:-2]).astype(int)

array([[0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 0, 0],
       [0, 0, 1, 1],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0]])

Updated preorder

- Clade below insertion point right-shift one position.
- All nodes after insertion point right-shift two positions.
- Connector inserted into first gap.
- New taxon inserted into the second gap.

In [63]:
order

array([ 0,  5,  3,  1,  4,  7, 17, 13,  9,  2, 10, 15, 14, 16, 18, 11,  8,
       12,  6])

In [64]:
order0[:6] + [17]

array([17, 22, 20, 18, 21, 24])

Updated sizes

- Sizes of nodes in the "spine" (from target's parent to root) +2.

In [65]:
sizes

array([19,  1,  1,  3,  1, 17,  1, 13,  1,  3,  1,  3,  1,  7,  1,  3,  1,
        9,  1])

Updated depths

- Entire clade below insertion point +1.

In [66]:
deeps

array([0, 3, 6, 2, 3, 1, 1, 2, 4, 5, 6, 3, 4, 4, 6, 5, 6, 3, 4])

Updated numbers of ancestor-descent pairs

- Nodes in the "spine" are updated.

In [67]:
pairs

array([68,  0,  0,  2,  0, 50,  0, 32,  0,  2,  0,  2,  0, 10,  0,  2,  0,
       18,  0])

Confirm that the numbers are correct.

In [68]:
_calc_sizes(n + 2, tree, order, sizes1 := np.empty(n + 2, dtype=int))
assert (sizes == sizes1).all()

In [69]:
_calc_deeps(n + 2, tree, order, deeps1 := np.empty(n + 2, dtype=int))
assert (deeps == deeps1).all()

In [70]:
_calc_pairs(n + 2, tree, order, sizes, pairs1 := np.empty(n + 2, dtype=int))
assert (pairs == pairs1).all()